## Importing required libraries

In [1]:
import os
import pickle
import catboost as cb
from time import time
import shap
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
import lightgbm as lgb
from sklearn.ensemble import AdaBoostRegressor
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import GradientBoostingRegressor

## Loading data

In [3]:
data = pd.read_excel(r"../Ea prediction dataset 302 reactionswithfnp.xlsx")
data = data.dropna()

## Feature extraction

In [4]:
x=data.iloc[:,7:]
y=data[["Ea"]]

In [9]:
x_base=x.iloc[:,:46]
x_finger=x[['Reactant₁712','Reactant₂712','Product₁712','Product₂712','Reactant₁795','Reactant₂795','Product₁795','Product₂795','Reactant₁474','Reactant₂474','Product₁474','Product₂474','Reactant₁1005','Reactant₂1005','Product₁1005','Product₂1005',]]
cor = x_base.corr('pearson')
# get upper triangle of correlation matrix
upper = cor.where(np.triu(np.ones(cor.shape), k=1).astype(np.bool_))

# find features with correlation greater than 0.90
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]

# drop highly correlated features
x_base.drop(to_drop, axis=1, inplace=True)

x = pd.concat([x_base,x_finger], axis=1, join='inner')

## Loading best ARE model

In [13]:
cat_model=cb.CatBoostRegressor(**{'iterations': 900,
 'learning_rate': 0.083,
 'depth': 2,
 'loss_function': 'RMSE',
 'silent': True,
 'subsample': 1.0,
 'colsample_bylevel': 0.9,
 'min_data_in_leaf': 9})

## Performing LOOCV

In [25]:
## Leave one out cross validation
from sklearn.model_selection import LeaveOneOut
loo = LeaveOneOut()
results_df = pd.DataFrame(columns=['Model','Train RMSE','Train R2','Train MAE', 'RMSE','R2','MAE'])
#  train the model on n-1 datapoints and test on the nth datapoint for all n datapoints
for train_index, test_index in loo.split(x):
    X_train, X_test = x.iloc[train_index], x.iloc[test_index]
    Y_train, Y_test = y.iloc[train_index], y.iloc[test_index]
    cat_model.fit(X_train, Y_train)
    cat_pred_train = cat_model.predict(X_train)
    cat_pred = cat_model.predict(X_test)
    cat_rmse_train = np.sqrt(mean_squared_error(Y_train, cat_pred_train))
    cat_r2_train = r2_score(Y_train,cat_pred_train)
    cat_mae_train= mean_absolute_error(Y_train,cat_pred_train)
    cat_rmse = np.sqrt(mean_squared_error(Y_test, cat_pred))
    cat_r2 = r2_score(Y_test,cat_pred)
    cat_mae= mean_absolute_error(Y_test,cat_pred)
    results_df = results_df._append({'Model': 'CatBoost','Train RMSE':cat_rmse_train,'Train R2':cat_r2_train,'Train MAE':cat_mae_train,'RMSE':cat_rmse,'R2':cat_r2,'MAE':cat_mae}, ignore_index=True)
#print average RMSE R2 and MAE
print(results_df.groupby('Model').mean())

          Train RMSE  Train R2  Train MAE      RMSE   R2       MAE
Model                                                             
CatBoost     0.12495  0.902223    0.09652  0.168656  NaN  0.168656


In [24]:
results_df

,Model,Train RMSE,Train R2,Train MAE,RMSE,R2,MAE
0,CatBoost,0.125643,0.900473,0.097188,0.381736,NaN,NaN
1,CatBoost,0.122697,0.905596,0.094847,0.519989,NaN,NaN
2,CatBoost,0.125471,0.901131,0.096772,0.031895,NaN,NaN
3,CatBoost,0.125182,0.902191,0.096855,0.209340,NaN,NaN
4,CatBoost,0.124069,0.903413,0.096701,0.285533,NaN,NaN
...,...,...,...,...,...,...,...
297,CatBoost,0.123563,0.904349,0.095309,0.112457,NaN,NaN
298,CatBoost,0.126633,0.899800,0.097387,0.154699,NaN,NaN
299,CatBoost,0.127108,0.898954,0.098304,0.110687,NaN,NaN
300,CatBoost,0.125660,0.901396,0.097356,0.035070,NaN,NaN
